# Saudi Medical Device Safety Alerts, 2023–2025
1,597 SFDA safety alerts for medical devices and supplies, published on Saudi Arabia's Open Data Platform.

**Questions:** which manufacturers and device types get the most alerts · what causes them · what corrective actions are taken · how manufacturers compare as suppliers

#Loading the SFDA safety alerts file and checking its size

In [1]:
# 1.1 Load the SFDA safety alerts file and check its size
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, re, os

BASE  = '/content/drive/MyDrive/pms-saudi-safety-alerts'
RAW   = f'{BASE}/data/raw'
CLEAN = f'{BASE}/data/clean'

alerts = pd.read_excel(f'{RAW}/Safety_Alerts_Data_for_Medical_Devices.xlsx')
alerts.columns = alerts.columns.str.strip()
print(alerts.shape)
print(alerts.columns.tolist())

Mounted at /content/drive
(1597, 7)
['Manufacturer Name', 'Name of the medical device', 'Affected devices', 'Cause of the problem Ar', 'Cause of the problem En', 'Corrective action', 'Creation date']


### 1.2 Check data quality
Before cleaning, we check three things: whether any fields are empty, the time period the alerts cover, and how many alerts SFDA published each year. This tells us whether the data is complete and whether alert numbers are rising or falling.

In [2]:
# 1.2 Check for missing values, the date range and alerts per year
alerts['Creation date'] = pd.to_datetime(alerts['Creation date'])
print(alerts.isna().sum())
print('First alert:', alerts['Creation date'].min().date(), '| Last alert:', alerts['Creation date'].max().date())
print(alerts['Creation date'].dt.year.value_counts().sort_index())

Manufacturer Name             0
Name of the medical device    0
Affected devices              0
Cause of the problem Ar       0
Cause of the problem En       0
Corrective action             0
Creation date                 0
dtype: int64
First alert: 2023-05-10 | Last alert: 2025-12-31
Creation date
2023    324
2024    632
2025    641
Name: count, dtype: int64


### 2.1 Clean the text
The cause and corrective-action fields contain leftover formatting codes (such as "_x000D_") and broken characters from the export. We remove these, tidy the spacing, and rename the columns to short, clear names so the rest of the analysis is easy to read.

In [3]:
# 2.1 Rename columns and clean the text (remove line-break codes and broken characters)
alerts = alerts.rename(columns={
    'Manufacturer Name': 'manufacturer', 'Name of the medical device': 'device',
    'Affected devices': 'affected_units', 'Cause of the problem Ar': 'cause_ar',
    'Cause of the problem En': 'cause_en', 'Corrective action': 'corrective_action',
    'Creation date': 'alert_date'})

def clean_text(t):
    t = str(t).replace('_x000D_', ' ')
    t = re.sub(r'\bx D\b', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return '' if t in ('-', 'nan') else t

for col in ['device', 'cause_en', 'cause_ar', 'corrective_action']:
    alerts[col] = alerts[col].map(clean_text)

print('Alerts with no English cause:', (alerts['cause_en'] == '').sum())
alerts[['device', 'cause_en']].head(3)

Alerts with no English cause: 16


,device,cause_en
0,"FLEX. GRASPING FORCEPS, FLEX. BIOPSY FORCEPS",as per attached FSN letter
1,"FLEX. GRASPING FORCEPS, FLEX. BIOPSY FORCEPS",As per attached FSN letter
2,"Ultraview SL (UVSL) Command Module, Model 91496",Command modules will display the following whe...


### 2.2 Standardise manufacturer names
The same company appears under several names (for example "PHILIPS", "PHILIPS MEDICAL SYSTEMS NEDERLAND B.V." and "PHILIPS MEDIZIN SYSTEME BOEBLINGEN GMBH"). We remove legal suffixes such as GmbH, Inc. and Ltd., then group known brands under one name, so each manufacturer is counted once. Subsidiaries keep their own name with the parent in brackets, e.g. "Covidien (Medtronic)".

In [4]:
# 2.2 Standardise manufacturer names so one company is counted once
name = (alerts['manufacturer'].str.upper()
        .str.replace(r'[^A-Z0-9& ]', ' ', regex=True)
        .str.replace(r'\b(B V|BV|GMBH|INC|LLC|LTD|LIMITED|CORP|CORPORATION|CO|COMPANY|SA|SAS|SRL|AG|PLC|AB|OY|NV|KG|SPA|PTY|KK|LP)\b', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True).str.strip())

brands = {r'^PHILIPS': 'Philips', r'^SIEMENS': 'Siemens Healthineers', r'^RANDOX': 'Randox',
          r'^GE\b|^GENERAL ELECTRIC': 'GE HealthCare', r'^ABBOTT': 'Abbott', r'^BECKMAN': 'Beckman Coulter',
          r'^MEDTRONIC': 'Medtronic', r'^ORTHO': 'Ortho Clinical Diagnostics', r'^OLYMPUS': 'Olympus',
          r'^MAQUET': 'Maquet (Getinge)', r'^GETINGE': 'Getinge', r'^BOSTON': 'Boston Scientific',
          r'^BECTON|^BD\b': 'BD', r'^COVIDIEN': 'Covidien (Medtronic)', r'^DATEX': 'Datex-Ohmeda (GE)',
          r'^WERFEN': 'Werfen', r'^BAXTER': 'Baxter', r'^DRAEGER|^DRAGER': 'Draeger', r'^BIOMERIEUX': 'bioMerieux',
          r'^STRYKER': 'Stryker', r'^B BRAUN': 'B. Braun', r'^SMITH NEPHEW|^SMITH & NEPHEW': 'Smith & Nephew',
          r'^SMITHS MEDICAL': 'Smiths Medical', r'^ROCHE': 'Roche', r'^FRESENIUS': 'Fresenius', r'^DEXCOM': 'Dexcom'}

alerts['manufacturer_std'] = name.str.title()
for pattern, brand in brands.items():
    alerts.loc[name.str.contains(pattern, regex=True), 'manufacturer_std'] = brand

print('Manufacturer names before:', alerts['manufacturer'].nunique(), '| after:', alerts['manufacturer_std'].nunique())
print(alerts['manufacturer_std'].value_counts().head(15))

Manufacturer names before: 538 | after: 339
manufacturer_std
Philips                       270
Siemens Healthineers           95
Randox                         74
GE HealthCare                  69
Abbott                         56
Beckman Coulter                54
Ortho Clinical Diagnostics     53
Medtronic                      52
Olympus                        34
Maquet (Getinge)               28
BD                             22
B. Braun                       18
Boston Scientific              18
Covidien (Medtronic)           16
Datex-Ohmeda (GE)              16
Name: count, dtype: int64


### 2.3 Convert affected units to numbers
The "Affected devices" field records how many units each alert covers. Some entries are "-" or text, and a few are negative, which is impossible. We convert the field to numbers and set invalid entries to empty, then look at the range of values.

In [5]:
# 2.3 Convert "affected units" to a number; "-", text and negative values become empty
alerts['affected_units'] = pd.to_numeric(alerts['affected_units'], errors='coerce')
alerts.loc[alerts['affected_units'] < 0, 'affected_units'] = None

print('Alerts with a unit count:', alerts['affected_units'].notna().sum(), 'of', len(alerts))
print(alerts['affected_units'].describe().round(0))

Alerts with a unit count: 1377 of 1597
count        1377.0
mean        25740.0
std        773464.0
min             0.0
25%             8.0
50%            41.0
75%           178.0
max      28675400.0
Name: affected_units, dtype: float64


### 2.4 Save the clean data
We save the cleaned alerts as a new file and confirm that no alerts were lost during cleaning. The original file stays untouched in the raw folder.

In [6]:
# 2.4 Save the clean file and confirm nothing was lost
os.makedirs(CLEAN, exist_ok=True)
assert len(alerts) == 1597, 'Row count changed during cleaning'
alerts.to_csv(f'{CLEAN}/alerts_clean.csv', index=False)
print('Saved data/clean/alerts_clean.csv with', len(alerts), 'alerts')

Saved data/clean/alerts_clean.csv with 1597 alerts
